In [3]:
import pandas as pd
import json
df = pd.read_csv(r"C:\Etudes\projects\data drills maven\Flatten the Stack\sales_orders.csv", parse_dates=["order_date"])
df["line_items"] = df["line_items"].apply(json.loads)
df.head()

,order_number,order_date,line_items,fulfillment
0,387005,2016-01-22,[{'product': {'product_name': 'MGS Age of Empi...,In store
1,395005,2016-01-30,[{'product': {'product_name': 'Contoso DVD 60 ...,In store
2,423011,2016-02-27,[{'product': {'product_name': 'Contoso 16GB Mp...,In store
3,600006,2016-08-22,[{'product': {'product_name': 'Contoso Water H...,Online
4,652002,2016-10-13,[{'product': {'product_name': 'Contoso Genuine...,In store


In [5]:
df_exploded = df.explode("line_items").reset_index(drop=True)
df_exploded.head()

,order_number,order_date,line_items,fulfillment
0,387005,2016-01-22,{'product': {'product_name': 'MGS Age of Empir...,In store
1,387005,2016-01-22,{'product': {'product_name': 'A. Datum Bridge ...,In store
2,387005,2016-01-22,{'product': {'product_name': 'WWI Desktop PC1....,In store
3,395005,2016-01-30,{'product': {'product_name': 'Contoso DVD 60 D...,In store
4,395005,2016-01-30,{'product': {'product_name': 'SV DVD 38 DVD St...,In store


In [6]:
df_json = pd.json_normalize(df_exploded["line_items"])
df_json.head()

,quantity,product.product_name,product.product_price
0,3,MGS Age of Empires II Gold Edition2009 E172,32.00
1,2,A. Datum Bridge Digital Camera M300 Pink,186.90
2,1,WWI Desktop PC1.80 E1801 Silver,269.90
3,5,Contoso DVD 60 DVD Storage Binder L20 Black,22.89
4,5,SV DVD 38 DVD Storage Binder E25 Silver,9.99


In [14]:
df_final = df_exploded.drop(columns=["line_items"]).join(df_json)
df_final.head()

,order_number,order_date,fulfillment,quantity,product.product_name,product.product_price
0,387005,2016-01-22,In store,3,MGS Age of Empires II Gold Edition2009 E172,32.00
1,387005,2016-01-22,In store,2,A. Datum Bridge Digital Camera M300 Pink,186.90
2,387005,2016-01-22,In store,1,WWI Desktop PC1.80 E1801 Silver,269.90
3,395005,2016-01-30,In store,5,Contoso DVD 60 DVD Storage Binder L20 Black,22.89
4,395005,2016-01-30,In store,5,SV DVD 38 DVD Storage Binder E25 Silver,9.99


In [12]:
df_final.columns

Index(['order_number', 'order_date', 'fulfillment', 'quantity', 'product_name',
       'product_price', 'total_sales'],
      dtype='str')

In [13]:
# df_final.columns = ['order_number', 'order_date', 'fulfillment', 'quantity',
#        'product_name', 'product_price']
df_final["total_sales"] = df_final["product_price"] * df_final["quantity"]
df_sales = (
    df_final
    .groupby("fulfillment")
    .agg(total_sales = ("total_sales", "sum"))
)
df_sales

,total_sales
fulfillment,
In store,183532.19
Online,18238.98
